In [5]:
# build_tile_index_and_map.py
import sys
import os
import rasterio
from pathlib import Path
import geopandas as gpd
from shapely.geometry import box
import folium

# --- CONFIG ---
SRC_DIR = Path(r"C:/Users/mgvhy/OneDrive - University of Missouri/scientific_data/glob_rad_annual")   # <-- update this
OUT_DIR = Path(r"C:/Users/mgvhy/OneDrive - University of Missouri/scientific_data/code_index")           # <-- update this
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_GPKG = OUT_DIR / "conus_tile_index.gpkg"
OUT_CSV = OUT_DIR / "conus_tile_index.csv"
EXPECTED_TILE_COUNT = 422

FILE_ATTRIBUTE_RECALL_ON_DATA_ACCESS = 0x00400000  # OneDrive "cloud-only, not downloaded" flag on Windows


def check_onedrive_placeholders(tif_files):
    """Warn if any files are still cloud-only placeholders (not synced locally)."""
    if not hasattr(os.stat_result, "st_file_attributes"):
        return  # not Windows, skip check
    cloud_only = []
    for f in tif_files:
        attrs = os.stat(f).st_file_attributes
        if attrs & FILE_ATTRIBUTE_RECALL_ON_DATA_ACCESS:
            cloud_only.append(f)
    if cloud_only:
        print(f"WARNING: {len(cloud_only)} of {len(tif_files)} files are still OneDrive "
              f"cloud-only placeholders (not downloaded locally). This will make reads "
              f"very slow or fail.")
        print("Fix: right-click the folder in File Explorer -> 'Always keep on this device', "
              "wait for sync to finish, then re-run.")
        print(f"First cloud-only file: {cloud_only[0]}")
        sys.exit(1)


def build_index():
    if not SRC_DIR.exists():
        raise SystemExit(f"SRC_DIR does not exist: {SRC_DIR}")

    tif_files = sorted(SRC_DIR.glob("*.tif"))
    print(f"Found {len(tif_files)} .tif files in {SRC_DIR}")
    if not tif_files:
        raise SystemExit("No .tif files found — check SRC_DIR and extension.")

    check_onedrive_placeholders(tif_files)

    records = []
    crs_seen = {}
    failed = []

    for i, tif in enumerate(tif_files):
        if i % 50 == 0:
            print(f"  processing {i}/{len(tif_files)}...")

        parts = tif.stem.split("_")
        if len(parts) < 2:
            print(f"  SKIP (unexpected filename format): {tif.name}")
            failed.append(tif.name)
            continue
        tile_id = parts[-2]  # glob_tile_000002_annual -> "000002"

        try:
            with rasterio.open(tif) as src:
                bounds = src.bounds
                crs = src.crs
                geom = box(*bounds)
        except Exception as e:
            print(f"  SKIP (could not read): {tif.name} — {e}")
            failed.append(tif.name)
            continue

        crs_str = crs.to_string()
        crs_seen[crs_str] = crs_seen.get(crs_str, 0) + 1
        records.append({"tile_id": tile_id, "geometry": geom, "_crs": crs_str})

    print(f"\nSuccessfully read {len(records)}/{len(tif_files)} tiles.")
    if failed:
        print(f"{len(failed)} files failed or were skipped: {failed[:10]}"
              f"{' ...' if len(failed) > 10 else ''}")

    if len(records) != EXPECTED_TILE_COUNT:
        print(f"WARNING: expected {EXPECTED_TILE_COUNT} tiles, got {len(records)}. "
              f"Investigate before publishing this index.")

    # --- verify all tiles share one CRS before trusting it for the whole GeoDataFrame ---
    print(f"\nDistinct CRS found across tiles: {crs_seen}")
    if len(crs_seen) > 1:
        raise SystemExit(
            "Multiple distinct CRSs found across tiles — the script assumed one shared "
            "CRS, which is false. Fix the mismatched tiles before building the index."
        )
    native_crs = list(crs_seen.keys())[0]

    for r in records:
        del r["_crs"]

    gdf = gpd.GeoDataFrame(records, crs=native_crs)

    # densify edges before reprojecting so curvature survives Albers -> WGS84
    gdf["geometry"] = gdf.geometry.segmentize(max_segment_length=5000)

    centroids_native = gdf.geometry.centroid
    centroids_wgs84 = centroids_native.to_crs("EPSG:4326")

    gdf = gdf.to_crs("EPSG:4326")
    gdf["centroid_lon"] = centroids_wgs84.x
    gdf["centroid_lat"] = centroids_wgs84.y

    bounds_wgs84 = gdf.geometry.bounds
    gdf["min_lon"] = bounds_wgs84["minx"]
    gdf["min_lat"] = bounds_wgs84["miny"]
    gdf["max_lon"] = bounds_wgs84["maxx"]
    gdf["max_lat"] = bounds_wgs84["maxy"]

    # sanity check: tile_id should be unique
    dupes = gdf["tile_id"][gdf["tile_id"].duplicated()].tolist()
    if dupes:
        print(f"WARNING: duplicate tile_id values found: {dupes}")

    gdf.to_file(OUT_GPKG, driver="GPKG")
    gdf.drop(columns="geometry").to_csv(OUT_CSV, index=False)
    print(f"\n{len(gdf)} tiles indexed -> {OUT_GPKG}")
    print(f"CSV -> {OUT_CSV}")

    return gdf
    
if __name__ == "__main__":
    build_index()
    print("\nDone.")

Found 422 .tif files in C:\Users\mgvhy\OneDrive - University of Missouri\scientific_data\glob_rad_annual
  processing 0/422...
  processing 50/422...
  processing 100/422...
  processing 150/422...
  processing 200/422...
  processing 250/422...
  processing 300/422...
  processing 350/422...
  processing 400/422...

Successfully read 422/422 tiles.

Distinct CRS found across tiles: {'EPSG:5070': 422}

422 tiles indexed -> C:\Users\mgvhy\OneDrive - University of Missouri\scientific_data\code_index\conus_tile_index.gpkg
CSV -> C:\Users\mgvhy\OneDrive - University of Missouri\scientific_data\code_index\conus_tile_index.csv

Done.
